In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
base = "/cosmos_storage/simulations/TNG_Family/MN5_resims/fiducial/hydro_output/"

sigma8 = 0.8159 #CHECK ME
ns     = 0.9667 #CHECK ME
tau    = 0.0965 #CHECK ME

_snap = 264
zoom_snap = bacco.Simulation(basedir=base, halo_file="groups_{:03d}/fof_subhalo_tab_{:03d}".format(_snap,_snap), dm_file="snapdir_{:03d}/snapshot_{:03d}".format(_snap,_snap), sim_format='TNG500', fixedPk=True, use_orphans=False,\
                        tau=tau, ns=ns, sigma8=sigma8, use_ids=False, tree_file="groups_{:03d}/subhalo_prog_{:03d}".format(_snap,_snap), numpart=4320**3)

In [ ]:
# Constants (CGS)
k_B = 1.380649e-16          # Boltzmann constant [erg/K]
m_p = 1.6726219e-24         # Proton mass [g]
X_H = 0.76                  # Hydrogen mass fraction
gamma = 5.0 / 3.0           # Adiabatic index

# UnitEnergy / UnitMass in CGS (given as 1e10 in the problem)
unit_energy_per_unit_mass = 1e10  # (km/s)^2 equivalent in cgs

Msun = 1.989e33       # [g]
kpc  = 3.0857e21      # [cm]
UnitDensity_cgs = (1e10 * Msun) / (kpc**3)   # ~6.77e-22 g/cm^3

def electron_number_density(ElectronAbundance, Density):
    """
    Convert TNG ElectronAbundance (= n_e / n_H) and Density [code units]
    into electron number density n_e [cm^-3].
    
    Density is the PartType0 'Density' field in units of (1e10 M_sun/h) / (ckpc/h)^3
    """
    rho_cgs = Density * UnitDensity_cgs # Convert to g/cm^3
    n_H = X_H * rho_cgs / m_p
    n_e = ElectronAbundance * n_H

    return n_e

def temperature_from_u_xe(InternalEnergy, ElectronAbundance):
    """
    Convert IllustrisTNG PartType0 InternalEnergy [code units] and
    ElectronAbundance into gas temperature [K].
    """
    mu = (4.0 / (1.0 + 3.0 * X_H + 4.0 * X_H * ElectronAbundance)) * m_p
    T = (gamma - 1.0) * InternalEnergy / k_B * unit_energy_per_unit_mass * mu
    
    return T

In [ ]:
U_int = zoom_snap.gas['add_prop']['InternalEnergy']
e_ab = zoom_snap.gas['add_prop']['ElectronAbundance']

In [ ]:
fig, ax = plt.subplots(dpi=100)

ax.set_xscale('log')
ax.hist(temperature_from_u_xe(U_int, e_ab), bins=np.logspace(2,9,100), log=True);

In [ ]:
def get_profiles(zoom, r_bins, ih, type='dm'):

    hpos = zoom.fof['halo_pos']
    r200 = zoom.fof['halo_r200c']
    r500 = zoom.fof['halo_r500c']

    _hpos = np.array([hpos[ih]])
    _r200 = np.array([r200[ih]])
    _r500c = np.array([r500[ih]])

    if type=='dm':
        _pos = zoom.dm['pos']
        _mass = np.ones_like(_pos[:,0])  * zoom_snap.header['ParticleMass'] * 1e10
    elif type=='gas':
        _pos = zoom.gas['pos']
        _mass = zoom.gas['mass'] * 1e10
        _Uint = zoom.gas['add_prop']['InternalEnergy']
        _eAb = zoom.gas['add_prop']['ElectronAbundance']
        _dens = zoom.gas['add_prop']['Density']
    elif type=='stars':
        _pos = zoom.stars['pos']
        _mass = zoom.stars['mass'] * 1e10
    elif type=='bh':
        _pos = zoom.bh['pos']
        _mass = zoom.bh['mass'] * 1e10

    mask = (_pos[:,0] > _hpos[0,0]-10) & (_pos[:,0] < _hpos[0,0]+10) & \
           (_pos[:,1] > _hpos[0,1]-10) & (_pos[:,1] < _hpos[0,1]+10) & \
           (_pos[:,2] > _hpos[0,2]-10) & (_pos[:,2] < _hpos[0,2]+10)

    t_ne = electron_number_density(_eAb[mask], _dens[mask])
    T = temperature_from_u_xe(_Uint[mask], _eAb[mask])

    y, c = ht.radial_profile_3d(_hpos, _pos[mask], t_ne * k_B * T, return_counts=True,\
                                period=500, rbins_normalized=r_bins, normalize_rbins_by=_r500c)
    rr = 10**((np.log10(r_bins[1:])+np.log10(r_bins[:-1]))*0.5)
    volume = 4 * np.pi / 3 *(r_bins[1:]**3 - r_bins[:-1]**3)

    dens1 = y * c / volume
    x1 = rr

    return x1, y

In [ ]:
r, dens = get_profiles(zoom_snap, np.logspace(-2,np.log10(3),20), 0, type='gas')

In [ ]:
fig, ax = plt.subplots(dpi=100, figsize=(5.5,5))

ax.set_xscale('log')
ax.set_yscale('log')

ax.plot(r, dens / P500(1e10 * zoom_snap.fof['halo_m500c'][0], 0) )
ax.set_xlabel('$r / r_{500,c}$')
ax.set_ylabel('Pressure')

In [ ]:
omega_m = zoom_snap.Cosmology.pars['omega_matter']
omega_de = zoom_snap.Cosmology.pars['omega_de']

def P500(M500, z):
    
    E_z = np.sqrt(omega_m * (1+z)**3 + omega_de)

    return 1.45e-11 * (M500 / 1e15)**(2/3) * E_z**(8/3) # erg/cm^3

In [ ]:
zoom = zoom_snap 

hpos = zoom.fof['halo_pos']
r200 = zoom.fof['halo_r200c']
r500 = zoom.fof['halo_r500c']

_hpos = np.array([hpos[ih]])
_r200 = np.array([r200[ih]])
_r500c = np.array([r500[ih]])

if type=='dm':
    _pos = zoom.dm['pos']
    _mass = np.ones_like(_pos[:,0])  * zoom_snap.header['ParticleMass'] * 1e10
elif type=='gas':
    _pos = zoom.gas['pos']
    _mass = zoom.gas['mass'] * 1e10
    _Uint = zoom.gas['add_prop']['InternalEnergy']
    _eAb = zoom.gas['add_prop']['ElectronAbundance']
    _dens = zoom.gas['add_prop']['Density']
elif type=='stars':
    _pos = zoom.stars['pos']
    _mass = zoom.stars['mass'] * 1e10
elif type=='bh':
    _pos = zoom.bh['pos']
    _mass = zoom.bh['mass'] * 1e10

mask = (_pos[:,0] > _hpos[0,0]-10) & (_pos[:,0] < _hpos[0,0]+10) & \
        (_pos[:,1] > _hpos[0,1]-10) & (_pos[:,1] < _hpos[0,1]+10) & \
        (_pos[:,2] > _hpos[0,2]-10) & (_pos[:,2] < _hpos[0,2]+10)

t_ne = electron_number_density(_eAb[mask], _dens[mask])
T = temperature_from_u_xe(_Uint[mask], _eAb[mask])


In [ ]:
np.median(t_ne)

In [ ]:
np.median(T/1e7)

In [ ]:
np.median(_dens)